CUSTOMER CHURN PREDICTION USING MACHINE LEARNING AND PYSPARK


# PROBLEM STATEMENT

Customer churn is one of the major challenges faced by telecom companies. 

The objective of this project is to analyze customer behavior and build machine learning models capable of predicting whether a customer is likely to leave the telecom service.

# IMPORTING LIBRARIES


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)


# LOADING DATASET

In [ ]:
df = pd.read_csv("Dataset.csv")

print("Dataset Loaded Successfully")

# DATA UNDERSTANDING

In [ ]:
df.head()


In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
df.info()

# DATA CLEANING

In [ ]:
df.isnull().sum()

In [ ]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [ ]:
df.dropna(inplace=True)

print("Missing Values Removed")

In [ ]:
print("Duplicate Rows:", df.duplicated().sum())

# EXPLORATORY DATA ANALYSIS

In [ ]:
sns.countplot(x="Churn", data=df)

plt.title("Customer Churn Distribution")

plt.show()

## Observation

The dataset shows moderate class imbalance between churned and retained customers. 

This imbalance is important because machine learning models may become biased toward the majority class during training.

In [ ]:
sns.countplot(
    x="gender",
    hue="Churn",
    data=df
)

plt.title("Gender vs Churn")

plt.show()

In [ ]:
sns.countplot(
    x="Contract",
    hue="Churn",
    data=df
)

plt.title("Contract Type vs Churn")

plt.xticks(rotation=15)

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(
    df["MonthlyCharges"],
    bins=30,
    kde=True
)

plt.title("Monthly Charges Distribution")

plt.show()

In [ ]:
plt.figure(figsize=(10,5))

sns.boxplot(
    x="Churn",
    y="tenure",
    data=df
)

plt.title("Tenure vs Churn")

plt.show()

In [ ]:
plt.figure(figsize=(14,10))

sns.heatmap(
    df.corr(numeric_only=True),
    cmap="coolwarm",
    annot=False
)

plt.title("Feature Correlation Heatmap")

plt.show()

## Interpretation

Feature importance analysis indicates that contract type, tenure, and monthly charges are the most influential variables affecting customer churn prediction.

In [ ]:
df.drop("customerID", axis=1, inplace=True)

# FEATURE ENGINEERING

In [ ]:
encoder = LabelEncoder()

for column in df.columns:
    if df[column].dtype == 'object' or df[column].dtype == 'string':
        df[column] = encoder.fit_transform(df[column])
        print("Encoding Completed")

In [ ]:
print(df.dtypes)

In [ ]:
X = df.drop("Churn", axis=1)

y = df["Churn"]

In [ ]:
print(X.select_dtypes(include="object").columns)

# FEATURE SCALING 

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
print("Scaling Completed")

# TRAIN TEST SPLIT

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

# MODEL BUILDING

In [ ]:
lr_model = LogisticRegression()

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

In [ ]:
print(
    "Logistic Regression Accuracy:",
    accuracy_score(y_test, lr_pred)
)

In [ ]:
dt_model = DecisionTreeClassifier(
    random_state=42
)

dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)

print(
    "Decision Tree Accuracy:",
    accuracy_score(y_test, dt_pred)
)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

In [ ]:
print(
    "Random Forest Accuracy:",
    accuracy_score(y_test, rf_pred)
)

In [ ]:
print(classification_report(
    y_test,
    rf_pred
))

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    
    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_test, rf_pred)
    ],
    
    "Precision": [
        precision_score(y_test, lr_pred),
        precision_score(y_test, dt_pred),
        precision_score(y_test, rf_pred)
    ],
    
    "Recall": [
        recall_score(y_test, lr_pred),
        recall_score(y_test, dt_pred),
        recall_score(y_test, rf_pred)
    ]
})

results

# MODEL EVALUATION

In [ ]:
cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Confusion Matrix")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
y_prob = rf_model.predict_proba(X_test)[:,1]

fpr, tpr, thresholds = roc_curve(
    y_test,
    y_prob
)

roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8,5))

plt.plot(
    fpr,
    tpr,
    label=f"AUC = {roc_auc:.2f}"
)

plt.plot([0,1],[0,1],'--')

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()

In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    x="Importance",
    y="Feature",
    data=importance
)

plt.title("Feature Importance")

plt.show()

In [ ]:
models = [
    "Logistic Regression",
    "Decision Tree",
    "Random Forest"
]

accuracies = [
    accuracy_score(y_test, lr_pred),
    accuracy_score(y_test, dt_pred),
    accuracy_score(y_test, rf_pred)
]

plt.figure(figsize=(8,5))

plt.bar(models, accuracies)

plt.title("Model Accuracy Comparison")

plt.ylabel("Accuracy")

plt.show()

# Business Insights

## Key Findings

- Customers with month-to-month contracts exhibit significantly higher churn rates compared to long-term contract users.

- Customers with shorter tenure are more likely to discontinue services, indicating onboarding-stage retention challenges.

- Higher monthly charges correlate with increased churn probability, suggesting pricing sensitivity among customers.

- Electronic check payment users demonstrate higher churn behavior than customers using automated payment methods.

- Contract type, tenure, and monthly charges emerged as the strongest predictive features in churn classification.

# PYSPARK IMPLEMENTATION

# CREATING SPARK SESSION

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [ ]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Customer Churn Analytics") \
    .getOrCreate()

# Loading dataset into Spark DataFrame

In [ ]:
spark_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") 

In [ ]:
print("PROJECT EXECUTED SUCCESSFULLY")

# Executive Summary

This project analyzes telecom customer churn behavior and develops machine learning models to predict customer attrition.

The workflow includes:
- data preprocessing,
- exploratory analysis,
- feature engineering,
- classification modeling,
- evaluation metrics,
- and scalable analytics using PySpark.

Random Forest achieved the strongest performance among evaluated models.